In [1]:
import sqlite3
import numpy as np

In [2]:
# Create a connection to the SQLite DB
conn = sqlite3.connect('vector-db.db')
# Create a cursor object to execute SQL Commands
cursor = conn.cursor()


In [3]:
  # Create a table for vector data

  cursor.execute(
      '''
      CREATE TABLE IF NOT EXISTS vectors(
          id INTEGER PRIMARY KEY,
          vector BLOB NOT NULL
      )
      '''
  )

In [4]:
# Generate some sample vector
vect1 = np.array([1.2,3.4,2.1,0.8])
vect2 = np.array([2.7,1.5,3.9,2.3])

In [5]:
  vect1.tobytes()  # Numpy array to bytestream

b'333333\xf3?333333\x0b@\xcd\xcc\xcc\xcc\xcc\xcc\x00@\x9a\x99\x99\x99\x99\x99\xe9?'

In [6]:
# Insert vector data into table
cursor.execute(
    '''
    INSERT INTO vectors(vector)
    VALUES(?)
    ''',
    (sqlite3.Binary(vect1.tobytes()),)
)

In [7]:
cursor.execute(
    '''
    INSERT INTO vectors(vector)
    VALUES(?)
    ''',
    (sqlite3.Binary(vect2.tobytes()),)
)

In [9]:
# Retreive data
cursor.execute(
    '''
    SELECT * FROM vectors
    '''
)

In [10]:
rows = cursor.fetchall()

In [11]:
rows

[(1,
  b'333333\xf3?333333\x0b@\xcd\xcc\xcc\xcc\xcc\xcc\x00@\x9a\x99\x99\x99\x99\x99\xe9?'),
 (2,
  b'\x9a\x99\x99\x99\x99\x99\x05@\x00\x00\x00\x00\x00\x00\xf8?333333\x0f@ffffff\x02@')]

In [12]:
vector = np.frombuffer(rows[0][1], dtype=np.float64)

In [13]:
vector

array([1.2, 3.4, 2.1, 0.8])

In [14]:
vectors = []
for row in rows:
    vector = np.frombuffer(row[1], dtype=np.float64)
    vectors.append(vector)

# Vector Similarity Search

In [15]:
query_vect = np.array([1.2,3.4,2.1,0.8])

In [16]:
cursor.execute("""
SELECT vector FROM vectors ORDER BY abs(vector - ?) ASC
""", (sqlite3.Binary(query_vect.tobytes()),))

# Finding the top one

In [17]:
res = cursor.fetchone()

# Most similar vector

In [18]:
np.frombuffer(res[0], dtype=np.float64)

array([1.2, 3.4, 2.1, 0.8])

In [19]:
conn.commit()
conn.close()